# Job flows (LBD) by ITL2 — cleaning

**Source:** `raw/Regional_Dynamism_LBD.xlsx`, *Site-level job flows by ITL2, 2000–2024* — produced from the **Longitudinal Business Database** (ONS, SN 9501) in the UK Data Service Secure Lab; aggregate, disclosure-checked output. Whole economy (incl. non-market), site (local-unit) level, on **ITL2 2021** boundaries (41 regions + "Unknown" = firms with no region).

**Six flows** (from the workbook notes): job creation from **entrant** enterprises' first site (`jc_entry`), from incumbents **opening new sites** (`jc_incumbent_ext`), from incumbents' **sites growing** (`jc_incumbent_int`); job destruction from incumbents' **sites shrinking** (`jd_incumbent_int`), incumbents **closing sites** (`jd_incumbent_ext`), and **exiting** enterprises (`jd_exit`).

**Rates — Davis & Haltiwanger definitions** (recomputed here so they are internally consistent):
- average employment $\bar E_t = (E_t + E_{t-1})/2$; JC rate $= JC_t/\bar E_t$; JD rate $= JD_t/\bar E_t$ (×100);
- **reallocation rate = JC rate + JD rate**; **net growth rate = JC rate − JD rate**;
- **excess reallocation = reallocation − |net growth|** — churn beyond what is needed to deliver the net change.

The workbook's own net rate divides by $E_{t-1}$ instead of $\bar E_t$ (and its excess reallocation inherits that), so it is replaced. Its JC/JD/reallocation rates match ours.

**Analysis window 2003–2024** (as in the proposal): 2000–2002 are LBD start-up years (no exits recorded in 2000; employment coverage jumps 2000→2001). All years are kept in the clean file with a `analysis_window` flag.

**Analysis** (summary statistics, regional variation, correlates, charts) is in `analysis/reallocation_v1.ipynb`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RAW   = BASE / "raw" / "Regional_Dynamism_LBD.xlsx"
DIR21 = BASE / "clean" / "itl2_2021"
OUT   = DIR21 / "job_flows_itl2.csv"
WINDOW = (2003, 2024)

FLOWS = ["jc_entry", "jc_incumbent_ext", "jc_incumbent_int", "jd_incumbent_int", "jd_incumbent_ext", "jd_exit"]
raw = pd.read_excel(RAW, header=5)
raw.columns = list(raw.columns[:18]) + ["net_emp_change_pct_wb", "excess_reallocation_rate_wb"]   # duplicate header in source
num = [c for c in raw.columns if c not in ("ITL221CD", "ITL221NM")]
assert not (raw[num].astype(str) == "c").any().any(), "suppressed cells present"
raw[num] = raw[num].apply(pd.to_numeric, errors="coerce")
raw = raw.rename(columns={"ITL221CD": "itl2_code", "ITL221NM": "itl2_name"})

spine21 = pd.read_csv(DIR21 / "geography_spine.csv")
reg = raw[raw.itl2_code != "Unknown"].copy()
assert set(reg.itl2_code) == set(spine21.itl2_code), "regions differ from the ITL2 2021 spine"
print(f"{reg.itl2_code.nunique()} regions x {reg.year.nunique()} years ({reg.year.min()}-{reg.year.max()}); "
      f"'Unknown' rows: {(raw.itl2_code == 'Unknown').sum()} (excluded from regional analysis)")
print(f"Unknown share of employment: {100 * raw[raw.itl2_code == 'Unknown'].emp.sum() / raw.emp.sum():.2f}%")

41 regions x 25 years (2000-2024); 'Unknown' rows: 25 (excluded from regional analysis)
Unknown share of employment: 0.65%


## 1. Recompute rates and check the flows

Checks: (i) JC and JD are the sums of their components; (ii) JC − JD reconciles with the change in employment (they can differ by disclosure rounding, and by sites changing region); (iii) our JC/JD/reallocation rates reproduce the workbook's.

In [2]:
reg = reg.sort_values(["itl2_code", "year"]).reset_index(drop=True)
assert np.allclose(reg[FLOWS[:3]].sum(axis=1), reg.total_jc) and np.allclose(reg[FLOWS[3:]].sum(axis=1), reg.total_jd)

def rates(d, by):
    """Davis-Haltiwanger rates from counts; d sorted by year within `by` groups."""
    d = d.copy()
    d["emp_lag"] = d.groupby(by).emp.shift() if by else d.emp.shift()
    d["emp_avg"] = (d.emp + d.emp_lag) / 2
    for c in FLOWS + ["total_jc", "total_jd"]:
        d[f"{c}_rate"] = 100 * d[c] / d.emp_avg
    d["reallocation_rate"] = d.total_jc_rate + d.total_jd_rate
    d["net_growth_rate"]   = d.total_jc_rate - d.total_jd_rate
    d["excess_reallocation_rate"] = d.reallocation_rate - d.net_growth_rate.abs()
    d["entry_share_of_jc"] = 100 * d.jc_entry / d.total_jc
    d["exit_share_of_jd"]  = 100 * d.jd_exit / d.total_jd
    return d

reg = rates(reg, "itl2_code")
reg["recon_gap_pct"] = 100 * ((reg.emp - reg.emp_lag) - (reg.total_jc - reg.total_jd)) / reg.emp_avg
print("Rates match workbook: JC", np.allclose(reg.total_jc_rate, reg.jc_rate, equal_nan=True),
      "| JD", np.allclose(reg.total_jd_rate, reg.jd_rate, equal_nan=True),
      "| reallocation", np.allclose(reg.reallocation_rate, raw.loc[raw.itl2_code != "Unknown"].sort_values(["itl2_code", "year"]).reallocation_rate.values, equal_nan=True))
big = reg[reg.recon_gap_pct.abs() > 1].sort_values("recon_gap_pct", key=abs, ascending=False)
print(f"\nΔemployment vs JC − JD: median |gap| {reg.recon_gap_pct.abs().median():.2f}% of avg employment; "
      f"region-years with |gap| > 1%: {len(big)}")
print(big[["itl2_code", "itl2_name", "year", "recon_gap_pct"]].round(1).head(12).to_string(index=False))

Rates match workbook: JC True | JD True | reallocation True

Δemployment vs JC − JD: median |gap| 0.10% of avg employment; region-years with |gap| > 1%: 20
itl2_code                          itl2_name  year  recon_gap_pct
     TLM8              West Central Scotland  2001          147.5
     TLN0                   Northern Ireland  2001          141.6
     TLM7                   Eastern Scotland  2001          140.6
     TLM5             North Eastern Scotland  2001          132.2
     TLM9                  Southern Scotland  2001          128.0
     TLM6              Highlands and Islands  2001          126.2
     TLI6               Outer London - South  2024           -4.2
     TLM9                  Southern Scotland  2008           -2.9
     TLI5 Outer London - East and North East  2024            2.8
     TLM8              West Central Scotland  2020            1.3
     TLI3                Inner London - West  2001           -1.2
     TLM5             North Eastern Scotland  2003  

## 2. Save the clean file

`clean/itl2_2021/job_flows_itl2.csv`: counts as numbers, recomputed rates, `analysis_window` flag (2003–2024) and `recon_gap_pct`. Region codes are ITL2 **2021** (`itl2_code`).

In [3]:
reg["analysis_window"] = reg.year.between(*WINDOW)
keep = (["itl2_code", "itl2_name", "year", "analysis_window", "n_sites", "emp", "emp_avg"] + FLOWS + ["total_jc", "total_jd"]
        + [f"{c}_rate" for c in FLOWS] + ["total_jc_rate", "total_jd_rate", "reallocation_rate", "net_growth_rate",
        "excess_reallocation_rate", "entry_share_of_jc", "exit_share_of_jd", "recon_gap_pct"])
reg[keep].to_csv(OUT, index=False, float_format="%.4f")
print(f"Saved -> {OUT}  ({len(reg)} rows)")

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/itl2_2021/job_flows_itl2.csv  (1025 rows)


## 3. Data caveats found (to raise with the LBD team)

- **2000–2002 are LBD start-up years** (no exits in 2000; Scottish and NI sites largely absent in 2000, so their 2001 flows don't reconcile with employment change). Excluded via `analysis_window`.
- **Northern Ireland 2003–2010 is erratic** (reallocation 15.6 → 10.9 → 42.3 → 11.4 → 47.2%…): a sawtooth that looks like register coverage, not behaviour. Treat NI's early years with caution or exclude them.
- **2010:** UK-wide spike in destruction from incumbents' sites shrinking (6.5 → 8.7% of employment) and closing — possibly a register/employment-measurement change rather than real churn.
- **2023:** jump in creation from incumbents opening new sites (3.0 → 4.4%) and a fall in the entrant share of creation — again worth checking for a register change.
- **The workbook's net rate** divides by lagged employment; replaced here by JC rate − JD rate (on average employment), which also fixes excess reallocation.
- Year effects absorb UK-wide steps (2010, 2023) in panel regressions; region-specific artefacts (NI) are not absorbed.